# Notebook zu: "Kryptographie Entschlüsselt": Algebra 2: Polynomringe und endliche Körper

© 2001–2026 Gerald Teschl (http://www.mat.univie.ac.at/~gerald/) und Susanne Teschl (http://staff.technikum-wien.at/~teschl/)

Dieses [Python](https://python.org)-Notebook ist als Ergänzung zu unserem [Springer](https://www.springer.de/)-Lehrbuch "[Kryptographie Entschlüsselt](https://www.mat.univie.ac.at/~gerald/ftp/book-kry/)" gedacht. Dort finden Sie auch weitere Erklärungen.

Der Code kann bequem von [PyPI](https://pypi.org/project/kryptools/) installiert werden.

## Polynomringe

Wir beginnen mit einer Klasse für Polynome. Ein Polynom wird dabei als Liste seiner Koeffizienten gespeichert.

Die Klasse definiert die Addition, Multiplikation und Division (mit Rest) und kann damit umgehen, wenn die Koeffizienten Elemente aus einem Ring sind.

In [1]:
from kryptools import Poly

Polynome werden durch die Liste der Koeffizienten erzeugt:

In [2]:
p = Poly([0, 1, 0, 1])
q = Poly([1, 1])
p, q

(x^3 + x, x + 1)

Mit folgendem Trick können Polynome in der gewohnten Form eingegeben werden:

In [13]:
x = Poly([0,1])

x**2 + x + 1

x^2 + x + 1

Polynomaddition und Polynommultiplikation:

In [3]:
p + q, p * q

(x^3 + 2 x + 1, x^4 + x^3 + x^2 + x)

Polynomdivision mit Rest:

In [4]:
p = Poly([0, -2, 0, 1, 3])
q = Poly([1, 0, 1])

p.divmod(q)

(3 x^2 + x - 3, -3 x + 3)

Probe

In [5]:
p == _[0] * q + _[1]

True

Alternativ funktioniert auch:

In [6]:
p // q, p % q

(3 x^2 + x - 3, -3 x + 3)

Die Koeffizienten können aus einem beliebigen Körper (Ring) sein. Für Polynome mit rationalen Koeffizienten kann die
Python-Klasse `Fraction` verwendet werden.

In [8]:
from fractions import Fraction

p = Poly([0, -2, 0, 1, 3], ring=Fraction)
Fraction(1 / 2) * p

3/2 x^4 + 1/2 x^3 - x

Wir können auch nachträglich den Ring ändern, indem wir mit dem Befehl `map` eine Funktion (z.B. den Konstruktor einer Klasse) auf alle Koeffizienten anwenden:

In [9]:
q.map(Fraction)
q / 4

1/4 x^2 + 1/4

Natürlich können die Koeffizienten auch aus $\mathbb{Z}_m$ sein. Die zugehörige Klasse `Zmod` kennen wir schon:

In [2]:
from kryptools import Zmod

Z_2 = Zmod(2)
p = Poly([0, -2, 0, 1, 3], ring=Z_2)
3 * p

x^4 + x^3

Um alle Teiler zu erhalten, muss das Polynom faktorisiert werden (das Paket kann nur Polynome über endlichen Körpern faktorisieren):

In [3]:
p.factor()

{x + 1: 1, x: 3}

Erweiterter Euklid'scher Algorithmus

In [11]:
def egcd(a: Poly, b: Poly) -> (Poly, Poly, Poly):
    """Perform the extended Euclidean agorithm for polynomials. Returns gcd, x, y such that a x + b y = gcd."""
    r0, r1 = a, b
    x0, x1, y0, y1 = Poly([1]), Poly([0]), Poly([0]), Poly([1])
    while r1:
        q, r = r0.divmod(r1)
        r0, r1 = r1, r
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return r0, x0, y0


a = Poly([1, -2, 0, 1])
b = Poly([-1, 0, 1])
egcd(a, b)

(- x + 1, 1, - x)

und die daraus abgeleitete Berechnung des multiplikativen Inversen:

In [12]:
def inverse(a: Poly, m: Poly) -> Poly:
    """Compute the multiplicative inverse in Z[x]_m. Returns b(x) such that a(x) * b(x) % m(x) = 1."""
    gcd, x, y = egcd(m, a)
    if gcd.degree() != 0:
        raise ValueError(f"{a} is not invertible mod {m}.")
    return Poly([1 / gcd.coeff[0]]) * y


a = Poly([0, -2, 0, 1, 3], ring=Fraction)
m = Poly([-1, 0, 1], ring=Fraction)
inverse(a, m)

1/8 x + 3/8

## Restklassenring

Um ein Polynom $p(x)$ bezüglich $m(x)$ zu reduzieren, können wir entweder den Rest der Division mit `%` berechnen oder `.mod(m)` verwenden:

In [4]:
p = Poly([1, 0, 0, 1]) # x^3 + 1
m = Poly([-1, 0, 1]) # x^2 - 1
p.mod(m)
p

x + 1

Dass alle Rechnungen mit einem Polynom modulo $m(x)$ durchgeführt werden, erreicht man wie folgt:

In [14]:
Poly([1, 0, 0, 1], modulus=[-1, 0, 1])

x + 1

Zum Beispiel können wir eine Multiplikationstabelle für den Restklassenring $\mathbb{Z}_2[x]_{x^3+x+1}$ erstellen:

In [8]:
from kryptools import Poly, Zmod

Z_2 = Zmod(2)
m = Poly([1,1,1], ring = Z_2)
poly_ring = [Poly([a, b], modulus = m) for b in Z_2 for a in Z_2]


[[p * q for p in poly_ring] for q in poly_ring]

[[0, 0, 0, 0], [0, 1, x, x + 1], [0, x, x + 1, 1], [0, x + 1, 1, x]]

Das multiplikative Inverse wird wie folgt berechnet:

In [9]:
e = Poly([1, 1], ring = Z_2, modulus = m)
e**-1

x

## Endliche Körper

Ob ein Polynom über einem endlichen Körper irreduzibel ist, können wir wie folgt feststellen:

In [10]:
from kryptools import Poly, Zmod

Z_2 = Zmod(2)

p = Poly([1,0,1,0], ring = Z_2)
p.is_irreducible()

False

Wir können es auch faktorisieren:

In [11]:
p.factor()

{x + 1: 2}

Zum Beispiel können wir den Restklassenring $\mathbb{Z}_2[x]_{x^8+x^4+x^3+x+1}$, mit dem bei AES gerechnet wird, implementieren:

In [18]:
from kryptools import Poly, Zmod

Z_2 = Zmod(2)
aes = [1, 1, 0, 1, 1, 0, 0, 0, 1]  # x^8 + x^4 + x^3 + x + 1


def PolyAES(c: list) -> "Poly":
    return Poly(c, ring = Z_2, modulus = aes)


p = PolyAES([1, 0, 1])  # x^2 + 1
q = PolyAES([0, 1, 0, 0, 1, 1, 1, 1])  # x^7 + x^6 + x^5 + x^4 + x

p * q

x^4 + x^2 + x + 1

Darstellung von Bytes als Polynome, deren Koeffizienten den Bits entsprechen:

In [19]:
def byte2poly(x: int, n: int = 8) -> "Poly":
    "Convert an integer to a Polynomial."
    return PolyAES(list(reversed([int(d) for d in str(format(x, "0" + str(n) + "b"))])))


def poly2byte(p: "Poly") -> int:
    "Convert a polynomial to the corresponding integer."
    s = 0
    for i in reversed(p.coeff):
        s = s * 2 + int(i)
    return s


poly2byte(byte2poly(2).inv())

141

In [28]:
byte2poly(0xa3) * byte2poly(0x02)

x^6 + x^4 + x^3 + x^2 + 1

In [29]:
hex(poly2byte(_))

'0x5d'

Mithilfe des ggT für Polynome

In [17]:
def poly_gcd(a: Poly, b: Poly) -> Poly:
    """Perform the Euclidean agorithm for polynomials."""
    r0, r1 = a, b
    while r1:
        q, r = r0.divmod(r1)
        r0, r1 = r1, r
    return r0


a = Poly([1, -2, 0, 1])  # x^3 -2 x + 1
b = Poly([-1, 0, 1])  # x^2 - 1
poly_gcd(a, b)

- x + 1

können wir den Rabin-Test implementieren:

In [18]:
def rabin_test(a: Poly, verbose: bool = False) -> bool:
    """Determine if a polynomial is irreducible over Z_p using a Rabin test."""
    gf = a[0].ring
    p = gf.n
    x = Poly([0, 1], ring=gf)  # x
    xp = Poly(p * [0] + [1], ring=gf, modulus = a.coeff)  # x^p modulo a to keep the polynomials small
    for k in range(1, a.degree() // 2 + 1):
        if k == 1:
            b = xp
        else:
            b **= p  # we compute x^{p^k} mod a recursively
        g = a.gcd(b - x)  # gcd with x^{p^k} - x
        if g.degree():
            if verbose:
                print(f"Factor found with k={k}: {g}")
            return False
    return True


a = Poly([1, 0, 1], ring=Z_2)

rabin_test(a, verbose=True)

Factor found with k=1: x + 1


False

Damit können wir das AES-Polynom als irreduzibel nachweisen:

In [19]:
a = Poly([1, 1, 0, 1, 1, 0, 0, 0, 1], ring=Z_2)
print(a)

rabin_test(a)

x^8 + x^4 + x^3 + x + 1


True

Die Anzahl der irreduziblen Polynome vom Grad $k$ über dem Körper $\mathbb{Z}_p$ kann mithilfe der Möbiusfunktion $\mu(n)$ angegeben werden. Für die Anzahl der irreduziblen Polynome vom Grad $8$ über dem Körper $\mathbb{Z}_2$ erhält man:

In [20]:
from kryptools import divisors, moebius_mu


def M(p: int, k: int) -> int:
    """Moreau's necklace-counting function."""
    return sum([moebius_mu(d) * p ** (k // d) for d in divisors(k)]) // k


M(2, 8)

30

Eine Liste dieser Polynome können wir wie folgt erhalten:

In [21]:
j = 0
for i in range(256):
    a = Poly(
        list(reversed([int(d) for d in str(format(256 + i, "0" + str(9) + "b"))])),
        ring=Z_2,
    )
    if rabin_test(a):
        j += 1
        print(j, ": ", a)

1 :  x^8 + x^4 + x^3 + x + 1
2 :  x^8 + x^4 + x^3 + x^2 + 1
3 :  x^8 + x^5 + x^3 + x + 1
4 :  x^8 + x^5 + x^3 + x^2 + 1
5 :  x^8 + x^5 + x^4 + x^3 + 1
6 :  x^8 + x^5 + x^4 + x^3 + x^2 + x + 1
7 :  x^8 + x^6 + x^3 + x^2 + 1
8 :  x^8 + x^6 + x^4 + x^3 + x^2 + x + 1
9 :  x^8 + x^6 + x^5 + x + 1
10 :  x^8 + x^6 + x^5 + x^2 + 1
11 :  x^8 + x^6 + x^5 + x^3 + 1
12 :  x^8 + x^6 + x^5 + x^4 + 1
13 :  x^8 + x^6 + x^5 + x^4 + x^2 + x + 1
14 :  x^8 + x^6 + x^5 + x^4 + x^3 + x + 1
15 :  x^8 + x^7 + x^2 + x + 1
16 :  x^8 + x^7 + x^3 + x + 1
17 :  x^8 + x^7 + x^3 + x^2 + 1
18 :  x^8 + x^7 + x^4 + x^3 + x^2 + x + 1
19 :  x^8 + x^7 + x^5 + x + 1
20 :  x^8 + x^7 + x^5 + x^3 + 1
21 :  x^8 + x^7 + x^5 + x^4 + 1
22 :  x^8 + x^7 + x^5 + x^4 + x^3 + x^2 + 1
23 :  x^8 + x^7 + x^6 + x + 1
24 :  x^8 + x^7 + x^6 + x^3 + x^2 + x + 1
25 :  x^8 + x^7 + x^6 + x^4 + x^2 + x + 1
26 :  x^8 + x^7 + x^6 + x^4 + x^3 + x^2 + 1
27 :  x^8 + x^7 + x^6 + x^5 + x^2 + x + 1
28 :  x^8 + x^7 + x^6 + x^5 + x^4 + x + 1
29 :  x^8 + x

### $GF(2^n)$

Da der Fall $GF(2^n)$ für die Kryptographie besonders wichtig ist, steht dafür die Klasse `GF2` zur Verfügung. Die Elemente werden als Zahlen aufgefasst und die Zuordnung zu Polynomen erfolgt über die Binärdarstellung.

In [12]:
from kryptools import GF2

gf = GF2(3)
[a.poly() for a in gf]

[0, 1, x, x + 1, x^2, x^2 + 1, x^2 + x, x^2 + x + 1]

Dabei wird automatisch ein irreduzibles Polynom aus einer Liste gewählt (alternativ kann es auch als Argument `modulus` bei der Erstellung angegeben werden):

In [15]:
gf.poly()

x^3 + x + 1

Damit können wir uns z.B. eine Additionstabelle, eine Multiplikationstabelle und eine Tabelle der multiplikativen Inversen erstellen:

In [24]:
[[a + b for a in gf] for b in gf]

[[0, 1, 2, 3, 4, 5, 6, 7],
 [1, 0, 3, 2, 5, 4, 7, 6],
 [2, 3, 0, 1, 6, 7, 4, 5],
 [3, 2, 1, 0, 7, 6, 5, 4],
 [4, 5, 6, 7, 0, 1, 2, 3],
 [5, 4, 7, 6, 1, 0, 3, 2],
 [6, 7, 4, 5, 2, 3, 0, 1],
 [7, 6, 5, 4, 3, 2, 1, 0]]

In [25]:
[[a * b for a in gf] for b in gf]

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 2, 4, 6, 3, 1, 7, 5],
 [0, 3, 6, 5, 7, 4, 1, 2],
 [0, 4, 3, 7, 6, 2, 5, 1],
 [0, 5, 1, 4, 2, 7, 3, 6],
 [0, 6, 7, 1, 5, 3, 2, 4],
 [0, 7, 5, 2, 1, 6, 4, 3]]

In [26]:
[a ** -1 for a in gf.star()]

[1, 5, 6, 7, 2, 3, 4]